<img src="https://backend.burla.dev/static/welcome.svg" width="45%">

#### This notebook **resizes 100 public images** into 3 sizes each, in parallel on 10 cloud workers, in about 3 minutes. <br/> It's only 4 steps — follow along!

### What is Burla?

Burla is the simplest way to run Python across hundreds or thousands of cloud machines.
One function — `remote_parallel_map` — fans your code out across isolated Docker containers, grows the cluster on demand, and streams results back. Missing packages on the workers? Burla auto-installs them.

No Dockerfiles, no cluster YAML, no queue. Just Python.  
Burla is free to try — this whole notebook runs on machines we spin up for you.

### What will this demo do?

You have 5M product images, satellite thumbnails, or training images on S3. You need 3 sizes (256 / 512 / 1024) of each. `Pillow` on one core is ~20ms per image — that's 27 hours single-threaded, even more once you factor in S3 bandwidth.

In this demo we download **100 public images** from `picsum.photos` (a free public image service), hand Burla 10 chunks of 10 URLs, and let each worker stream-resize its images into 3 sizes. The worker returns thumbnail bytes for the smallest size so we can show a montage inline.

The full-scale version in [`main.py`](https://github.com/Burla-Cloud/image-dataset-resize/blob/main/main.py) does this for 5M S3 images across 5,000 workers, writing back to S3.

## 1)&nbsp; Boot some machines (10+ CPUs):
&nbsp;&nbsp;&nbsp;&nbsp;Sign in using the button below, then hit the **`⏻ Start`** button on your dashboard homepage.  
&nbsp;&nbsp;&nbsp;&nbsp;This will boot a small cluster in Google Cloud for you. Burla is free to try so this is on us!

&nbsp;&nbsp;
<a href="https://login.burla.dev">
    <img src="https://i.ibb.co/QFNncHcp/Clean-Shot-2026-03-19-at-18-13-07.jpg" width="28%">
</a>

## 2)&nbsp; Install the Python package:

In [ ]:
!uv pip install burla

## 3)&nbsp; Authorize this computer:

In [ ]:
!burla login

## 4)&nbsp; Build the URL list

`picsum.photos` serves a different 800x800 image for each seed ID. 100 URLs = 100 unique images.

In [ ]:
urls = [f"https://picsum.photos/seed/burla{i:03d}/800/800" for i in range(100)]
print(f"{len(urls)} image URLs")

## 5)&nbsp; Resize on 10 workers in parallel 🚀

Each worker gets 10 URLs. For each image it downloads, generates 3 resized versions (256/512/1024), and returns the **256-px thumbnail as PNG bytes** + per-image metadata. Burla auto-installs `Pillow` on the workers.

In [ ]:
from PIL import Image, ImageOps  # noqa: F401  # top-level import -> Burla installs Pillow on workers
from burla import remote_parallel_map

CHUNK = 10
chunks = [urls[i : i + CHUNK] for i in range(0, len(urls), CHUNK)]
print(f"{len(urls)} images in {len(chunks)} chunks of {CHUNK}")


def resize_chunk(image_urls: list[str]) -> list[dict]:
    import io
    import urllib.request
    from PIL import Image, ImageOps

    out = []
    for url in image_urls:
        try:
            with urllib.request.urlopen(url, timeout=20) as r:
                body = r.read()
            img = Image.open(io.BytesIO(body))
            img = ImageOps.exif_transpose(img).convert("RGB")
            w, h = img.size

            sizes = {}
            for size in (256, 512, 1024):
                resized = img.copy()
                resized.thumbnail((size, size), Image.Resampling.LANCZOS)
                buf = io.BytesIO()
                resized.save(buf, format="JPEG", quality=85, optimize=True, progressive=True)
                sizes[size] = len(buf.getvalue())

            # return the 256px thumbnail bytes so we can show a montage
            thumb = img.copy()
            thumb.thumbnail((128, 128), Image.Resampling.LANCZOS)
            thumb_buf = io.BytesIO()
            thumb.save(thumb_buf, format="PNG", optimize=True)

            out.append({
                "url": url,
                "orig_w": w,
                "orig_h": h,
                "size_256_bytes": sizes[256],
                "size_512_bytes": sizes[512],
                "size_1024_bytes": sizes[1024],
                "thumb_png": thumb_buf.getvalue(),
            })
        except Exception as e:
            out.append({"url": url, "error": str(e)})

    return out


chunk_results = remote_parallel_map(
    resize_chunk,
    chunks,
    func_cpu=1,
    func_ram=2,
    grow=True,
)

rows = [r for chunk in chunk_results for r in chunk]
print(f"processed {len(rows)} images ({sum(1 for r in rows if 'error' in r)} errors)")

### What just happened?

You just resized 100 images × 3 sizes = 300 re-encodes on 10 cloud workers, without installing `Pillow` locally, setting up S3, or building a Dockerfile. The same shape scales to the `main.py` version: 5M images, 5,000 workers, reading from and writing to your own S3 bucket.

### Inspect the results

Show a 10×10 grid of the 128px thumbnails returned by the workers, plus per-size compression ratios.

In [ ]:
import io
import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image

ok = [r for r in rows if "error" not in r]
print(f"showing {min(100, len(ok))} thumbnails")

fig, axes = plt.subplots(10, 10, figsize=(10, 10))
for ax, r in zip(axes.flat, ok[:100]):
    ax.imshow(Image.open(io.BytesIO(r["thumb_png"])))
    ax.axis("off")
for ax in axes.flat[len(ok):]:
    ax.axis("off")
plt.subplots_adjust(wspace=0.02, hspace=0.02)
plt.show()

df = pd.DataFrame([r for r in rows if "error" not in r])
print()
print("avg size (KB) per output resolution:")
for col in ("size_256_bytes", "size_512_bytes", "size_1024_bytes"):
    print(f"  {col:20s} {df[col].mean() / 1024:7.1f} KB")

### Try this next

- Crank it: 10K URLs, 100 workers, `max_parallelism=100`. Same code, same runtime shape.
- Point at your own S3 bucket — drop `boto3` into the worker and `s3.get_object` / `s3.put_object` the resized outputs.
- Add WebP or AVIF output for ~30% smaller files.
- Replace resize with crop-to-face using `face_recognition` or `mediapipe`.

## Thank you for trying Burla! ❤️

### Run the full 5M-image version

See [`main.py`](https://github.com/Burla-Cloud/image-dataset-resize/blob/main/main.py) in this repo — 5 million S3 images, 5,000 workers, writing resized outputs back to S3.

### Run this on your laptop 🧑‍💻

1. `pip install burla`
2. `burla login`
3. Run the same example code from above ☝️

Congrats! Your laptop now effectively has 1000+ CPUs, and 4000G of RAM &nbsp; : )

<br/>

### Any way we can help? Lets chat!

Schedule a meeting: https://cal.com/jakez  
Send me an email: jake@burla.dev